<a href="https://colab.research.google.com/github/AktanM11/AI-OI/blob/main/5DAY_4Week_redemption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U FlagEmbedding

In [ ]:
!pip install qdrant-client

In [ ]:
!pip install langgraph

In [ ]:
!pip install langchain-groq

In [6]:
import qdrant_client
from qdrant_client import QdrantClient
from google.colab import userdata

In [11]:
import FlagEmbedding

In [8]:
qdrant_client = QdrantClient(url=userdata.get('QDRANT_URL'), api_key=userdata.get('QDRANT_API_KEY')) # Векторная база данных

In [1]:
collection_name = "policies_collection_bge"

In [ ]:
bge_model = FlagEmbedding.FlagAutoModel.from_finetuned('BAAI/bge-m3', use_fp16=True) # Модель embedding

In [13]:
test_text = "тест размерности"
output = bge_model.encode(test_text, return_dense=True, return_sparse=False, return_colbert_vecs=False)

dense_vector = output['dense_vecs']

dimension = len(dense_vector)
print(dimension)

1024


In [16]:
vector_dimension = 1024
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
BATCH_SIZE = 32

In [10]:
path = "policies.jsonl"

In [28]:
text = "о магазине О!" # Запрос

In [15]:
from qdrant_client.models import PointStruct, VectorParams, Distance
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_dimension, distance=Distance.COSINE),
)

True

In [21]:
from typing import List, Dict, Generator
def recursive_chunking(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
    """
    Разбивает текст на куски заданного размера с контролируемым перекрытием.
    Рекурсивно подстраивается, если кусок текста не требует деления.
    """
    if len(text) <= chunk_size:
        return [text] if text.strip() else []


    chunk = text[:chunk_size]

    slice_zone = chunk[-overlap:]
    last_space = slice_zone.rfind(" ")

    if last_space != -1:
        cut_index = (chunk_size - overlap) + last_space + 1
    else:
        cut_index = chunk_size

    final_chunk = text[:cut_index].strip()

    remaining_text = text[cut_index - overlap:]

    return [final_chunk] + recursive_chunking(remaining_text, chunk_size, overlap)

In [22]:
import json
import uuid
import os
def read_policies(file_path: str) -> Generator[str, None, None]:
    """Построчно читает jsonl файл и возвращает сырой текст политики."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Файл {file_path} не найден")

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                if "content" in data:
                    yield data["content"]

In [26]:
points_batch = []
total_chunks = 0
for raw_text in read_policies(path):
    chunks = recursive_chunking(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

    for chunk in chunks:
        if not chunk.strip():
            continue

        text_for_embedding = f"passage: {chunk}"
        vector = bge_model.encode(text_for_embedding)['dense_vecs']

        point = PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={"page_content": chunk}
        )

        points_batch.append(point)
        total_chunks += 1

        if len(points_batch) >= BATCH_SIZE:
            qdrant_client.upsert(
                collection_name=collection_name,
                points=points_batch
            )
            print(f"Загружен батч из {len(points_batch)} точек. Всего обработано: {total_chunks}")
            points_batch = []

if points_batch:
    qdrant_client.upsert(
        collection_name=collection_name,
        points=points_batch
    )
    print(f"Загружен финальный батч из {len(points_batch)} точек. Всего обработано: {total_chunks}")

print(f"Конвейер успешно завершен. Всего точек в базе: {total_chunks}")

Загружен батч из 32 точек. Всего обработано: 32
Загружен батч из 32 точек. Всего обработано: 64
Загружен батч из 32 точек. Всего обработано: 96
Загружен батч из 32 точек. Всего обработано: 128
Загружен батч из 32 точек. Всего обработано: 160
Загружен финальный батч из 17 точек. Всего обработано: 177
Конвейер успешно завершен. Всего точек в базе: 177


In [29]:
user_question = bge_model.encode(text)['dense_vecs'] # Зашифровка запроса в вектор (embedding)

In [30]:
results = qdrant_client.query_points(
    collection_name=collection_name,
    query=user_question,
    limit=3
)
# Поиск через Qdrant (ищет не модель, а по сути Qdrant считает косинусное расстояние между вектором запроса и векторами в коллекции
# Без фильтра

In [32]:
points_results = results.points
print(f"Запрос: {text}\n")
for idx, hit in enumerate(points_results):
    print(f"Ответ {idx+1}: {hit.payload.get('category')}")
    print({hit.payload.get('page_content')})
    print(f"{hit.payload.get('source_file')}\n")

Запрос: о магазине О!

Ответ 1: None
{'ы: Пн-Вс(09:00-19:00) город ОшОш O!Store Ареопаг: ул.Масалиева 10, График работы: Пн-Вс(09:00-21:00) город ОшМакском: ул. А. Шакирова 30 - торговый центр «Maxcom Urban Mall», 1-ый этаж, График работы: Пн-Вс(10:00-21:00)Перерыв Пн-Вс(13:00-14:00) город ОшОш КУУ: ул. Насирдин Исанова, 59а, ориентир: напротив "Кыргызско-Узбекского университета", График работы: Пн-Вс(09:00-19:00)Перерыв Сб-Вс(13:00-14:00) город ОшОш O!Store 1: O!Store ул. Курманжан Датка, 256 (бывшая Араванская), напротив'}
None

Ответ 2: None
{' рассрочку. Постоянные акции и скидки. Покупать в O!Store – выгодно!'}
None

Ответ 3: None
{'ИНТЕРНЕТ-МАГАЗИН OSTORE.KG O!Store – интернет-магазин, у которого более 110 филиалов по всему Кыргызстану. В магазинах огромный выбор смартфонов мировых брендов. Есть всё, для подключения к скоростному мобильному интернету: роутеры, винглы, семейные комплекты для дома! Больше 20 000 аксессуаров для мобильных устройств: кабели, переходники, адаптеры, ак

In [33]:
import langchain_groq
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key=userdata.get('GROQ_API_KEY'),
    model="llama-3.3-70b-versatile"   # моделька
)

In [51]:
import langgraph
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
def retrieval_node(state: MessagesState) -> dict:
    """
    Node is used for retrieving the data from the Qdrant database by using encoder BGE-M3
    """
    user_question = state["messages"][-1].content   # из списка сообщений берёт последнее сообщение (запрос)
    retrieved_chunks = []
    try:
        query_vector = bge_model.encode(user_question)['dense_vecs']
        results = qdrant_client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=3
        )
        if results.points:
            chunk_text = results.points[0].payload.get('page_content')
            retrieved_chunks.append(f"Context chunk: {chunk_text}")
    except Exception as e:
        print(f"Error: {e}")

    context = "\n\n".join(retrieved_chunks) if retrieved_chunks else "Контекст не найден."
    return {"messages": [AIMessage(content=context, name="context_holder")]}

# To save the content of the search of embedded vector by BGE-M3

In [59]:
def generation_node(state: dict):
    """
    Node is used for generating the output by LLM
    """
    messages = state["messages"]    # весь список сообщений
    context = messages[-1].content  # данные из "извлечения" после поиска в Qdrant по векторной бд
    user_question = messages[-2].content # запрос пользователя

    system_prompt = (
        "Вы — корпоративный AI-ассистент поддержки сотрудников O!Store.\n"
        "Ваша задача — строго отвечать на вопросы пользователя на основе предоставленного контекста.\n"
        "Если в контексте нет четкого ответа на вопрос или модели поиска вернули ошибочные данные, "
        "вы ОБЯЗАНЫ строго ответить: 'К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store.'\n"
        "Не придумывайте факты, не используйте внешние знания, которых нет в тексте.\n\n"
        f"ПРЕДОСТАВЛЕННЫЙ КОНТЕКСТ:\n{context}"
    )

    response = llm.invoke([       # отправляем в LLM(llama3.3) два сообщения
        SystemMessage(content=system_prompt),   # промпт + контекст
        HumanMessage(content=user_question)     # запрос
    ])

    return {
        "messages": [AIMessage(content=response.content)]
    }

workflow = StateGraph(MessagesState)
workflow.add_node("generator", generation_node)
workflow.add_node("retriever", retrieval_node)
workflow.add_edge(START, "retriever")
workflow.add_edge("retriever", "generator")
workflow.add_edge("generator", END)
app = workflow.compile()

In [53]:
def run_pipeline(query: str) -> str:
    """Запускает пайплайн и возвращает финальный ответ."""
    init_state = {"messages": [HumanMessage(content=query)]}
    result = app.invoke(init_state)

    for msg in reversed(result["messages"]):
        if isinstance(msg, AIMessage) and not msg.name:
            return msg.content
    return result["messages"][-1].content

In [57]:
test_queries = [
    "Какие документы нужны для оформления гарантийного ремонта?",
    "Как оформить корпоративный заказ?",
    "Что делать если покупатель хочет обменять телефон?",
    "Каковы условия постгарантийного обслуживания?"
]

In [55]:
answer = run_pipeline("Каков порядок возврата товара?")
print(answer)

Для осуществления возврата товара абонент обязан предъявить заключение сервисного центра Принципала о том, что данный конкретный товар соответствует условиям обмена или возврата товара. После этого возврат денежных средств юридическому лицу при оплате покупки безналичным способом производится безналичным способом на расчетный счет в течение 10 (десяти) банковских дней после обращения.


In [58]:
for query in test_queries:
    print(f"\nЗапрос: {query}")

    baseline_ans = run_pipeline(query)
    print(f"[Baseline] {baseline_ans[:250]}...")


Запрос: Какие документы нужны для оформления гарантийного ремонта?
[Baseline] Для оформления гарантийного ремонта необходимы следующие документы: 
1. Правильно и без помарок и исправлений заполненный гарантийный талон, в котором должны быть указаны модель и серийный номер изделия, дата продажи и печать торгующей организации; 
...

Запрос: Как оформить корпоративный заказ?
[Baseline] К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store....

Запрос: Что делать если покупатель хочет обменять телефон?
[Baseline] Если покупатель хочет обменять телефон, он должен обратиться в O!Store для 4G-устройств компании или в Сервис-центр для телефонов, при этом товар должен сохранить свой товарный вид и его неработоспособность должна быть подтверждена. Кроме того, покуп...

Запрос: Каковы условия постгарантийного обслуживания?
[Baseline] К сожалению, я не могу ответить на этот вопрос на основе имеющихся регламентов O!Store....
